
# Geologic Rules → Code Whiteboard (Fluvial)

Purpose: Transparently link each geologic principle to the exact code that implements it. For each rule we describe (a) the geologic intent, (b) the algorithmic logic, and (c) where to improve. Use this as a lecture/whiteboard: read the markdown for rationale, and inspect the code cells (loaded via `inspect.getsource`) to trace implementation details.

Scope: fluvial environments (meandering, braided, anastomosing), stacked packages, sedimentary overlays, and metrics. Aeolian/estuarine are placeholders until their milestones land.


In [ ]:
import inspect
from analog_image_generator import geologic_generators as gg
from analog_image_generator import stacked_channels as sc
from analog_image_generator import stats



## Meandering — rules, logic, and code

- **Principle: Sinuous belt from control points** (anchor-fluvial-meander-centerline)
  - Geologic intent: Meandering rivers migrate laterally with wavelength tied to belt width; control points set curvature.
  - Logic: `meander_centerline` seeds control points across the belt width, adds random drift, applies a sinusoidal modulation, and interpolates a continuous centerline.
  - Improvement ideas: Tie amplitude to discharge/sinuosity ranges from literature; add cut-bank stabilization constraints.
- **Principle: Variable bankfull width** (anchor-fluvial-variable-width)
  - Geologic intent: Width varies along the belt; relates to discharge, confinement, and local curvature.
  - Logic: `meander_variable_channel` builds a column-wise width curve (min→max with noise), then creates a boolean mask around the centerline based on half-width.
  - Improvement ideas: Couple width to curvature (narrow in tight bends), and to synthetic discharge.
- **Principle: Levees as rims** (anchor-fluvial-levees)
  - Geologic intent: Overbank deposition forms levee rims adjacent to channels.
  - Logic: `add_levees` dilates/gaussian-blurs the channel mask and subtracts the channel to yield rims; number of iterations controls levee width/height proxy.
  - Improvement ideas: Thickness taper away from channel; asymmetry for point bars vs cut banks.
- **Principle: Scroll-bar banding** (anchor-fluvial-scroll-bars)
  - Geologic intent: Point-bar accretion leaves scrolls with quasiperiodic spacing.
  - Logic: `add_scroll_bars` computes distance-to-channel and applies a cosine with wavelength `scroll_lambda_px` to modulate bar bands.
  - Improvement ideas: Vary wavelength with curvature radius; add stochastic breaks.
- **Principle: Oxbow/clay plugs** (anchor-fluvial-oxbow)
  - Geologic intent: Neck cutoffs leave oxbows; often clay plugged.
  - Logic: `add_oxbow` randomly stamps circular scars near the centerline with probability `oxbow_probability` and random radius.
  - Improvement ideas: Trigger oxbow probability from curvature thresholds; tie plug texture to floodplain fines.
- **Composition & overlays** (anchor-fluvial-compose)
  - Logic: `compose_meandering` blends channel/levee/scroll/oxbow/floodplain grayscale + noise, then `apply_sedimentary_overlays` adds facies textures (cross-bedding, ripple, lateral accretion, fining-upward, mudstone) scaled by user strengths.
  - Improvement ideas: Calibrate overlay strengths to literature ranges; add clay clast scars or crevasse-splay lenses.

**Key params**: `n_control_points`, `amplitude_range`, `drift_fraction`, `channel_width_min/max`, `scroll_lambda_px`, `oxbow_probability`, overlay strengths (`cross_bed_strength`, `ripple_strength`, `ripple_wavelength_px`, `fining_strength`, `mudstone_strength`, `lateral_accretion_strength`).


In [ ]:
inspect.getsource(gg.meander_centerline)


## Braided — rules, logic, and code

- **Principle: Multi-threads across belt** (anchor-fluvial-braided-threads)
  - Geologic intent: Braided systems host multiple active channels with bar-braid bars; widths and amplitudes vary.
  - Logic: `braided_threads` seeds several sinusoidal centerlines with random amplitude/frequency, clips widths to realistic ranges, and sums to a channel mask.
  - Improvement ideas: Couple thread count/width to discharge slope; add bank erodibility control.
- **Principle: Bar spacing ≈ 4–5× mean width** (anchor-fluvial-bar-spacing)
  - Logic: `seed_bars` computes bar centers along threads, using `bar_spacing_factor * width` to place ellipsoids gated by channel mask.
  - Improvement ideas: Vary spacing with local thread width; include bar amalgamation probability.
- **Principle: Chutes cross-cut bars** (anchor-fluvial-chutes)
  - Logic: `add_chutes` draws thin linear cuts connecting thread centerlines; count proportional to `chute_frequency`.
  - Improvement ideas: Angle distribution tied to slope; chute width tied to bar thickness.
- **Composition & overlays** (anchor-fluvial-braided-compose)
  - Logic: `compose_braided` blends channel/bar/chute/floodplain grayscale + noise; `apply_sedimentary_overlays` adds cross-bedding (trough), ripple, lateral accretion, fining/mudstone as applicable.

**Key params**: `thread_count`, `mean_thread_width`, `bar_spacing_factor`, `chute_frequency`, overlay strengths as above.


In [ ]:
inspect.getsource(gg.braided_threads)


## Anastomosing — rules, logic, and code

- **Principle: Narrow, stable branches** (anchor-fluvial-anasto-paths)
  - Geologic intent: Anastomosing rivers have multiple stable narrow channels separated by wetlands.
  - Logic: `anasto_paths` builds low-sinuosity branches with controlled width; masks combined into a branch_channel.
- **Principle: Levees for narrow channels** (anchor-fluvial-anasto-levees)
  - Logic: `add_levees_narrow` dilates/filters branch masks with width/height scaling.
- **Principle: Wetlands from distance + base quantile** (anchor-fluvial-anasto-marsh)
  - Logic: `make_marsh` uses distance to channels and a quantile on base grid to fill marsh/overbank/water masks.
- **Principle: Fans at levee breaches** (anchor-fluvial-anasto-fans)
  - Logic: `_select_breach_points` picks breach locations; `seed_fans` paints fans with length `fan_length_px`.
- **Composition & overlays** (anchor-fluvial-anasto-compose)
  - Logic: `compose_anasto` blends branch/levee/marsh/fan/overbank gray + noise; overlays as above.

**Key params**: `branch_count`, `levee_width_px`, `levee_height_scale`, `marsh_fraction`, `fan_length_px`, overlay strengths.


In [ ]:
inspect.getsource(gg.anasto_paths)


## Stacked packages — rules, logic, and code

- **Principle: Toggle single vs stacked** (anchor-fluvial-stacked-packages)
  - Logic: `gg.generate_fluvial` routes to `sc.build_stacked_fluvial` when `mode='stacked'`.
- **Principle: Ordered packages & metadata** (anchor-fluvial-stack-sequence)
  - Logic: `sc.sequence_packages` cycles styles (`package_styles`), thickness, relief, and erosion depth per package; records package_id_map and metadata.
- **Principle: Relief/erosional trimming** (anchor-fluvial-stack-relief)
  - Logic: `apply_relief_slice`/`cut_erosional_surface` impose relief between packages before compositing.
- **Metadata export**: `realization_metadata['stacked_packages']` stores package stats and mask means.

**Key params**: `package_count`, `package_mix`, `package_relief_px`, `package_erosion_depth_px`, `package_thickness_px`, `stack_seed`.


In [ ]:
inspect.getsource(sc.build_stacked_fluvial)


## Sedimentary overlays — rules, logic, and code

- **Channel-fill sandstone** (anchor-fluvial-channel-fill)
  - Logic: `channel_fill_sandstone` thickens channel interiors; enriches mineralogy metadata.
- **Cross-bedding (trough/planar)** (anchor-fluvial-cross-bedding)
  - Geologic intent: Sets internal lamination; trough for braided, planar for meander/anasto.
  - Logic: `apply_cross_bedding` sinusoidal bands with orientation, scaled into channel_fill.
- **Ripple marks** (anchor-fluvial-ripple-marks)
  - Logic: `ripple_mark_texture` sinusoidal texture on overbank; wavelength tunable via `ripple_wavelength_px`.
- **Lateral accretion surfaces** (anchor-fluvial-lateral-accretion)
  - Logic: distance gradient → Sobel edge → jitter; scaled by `lateral_accretion_strength`.
- **Fining-upward + mudstone** (anchor-fluvial-fining-upward)
  - Logic: distance-to-channel → fining mask; overbank mask → mudstone with noise; both scaled by strengths.
- **Petrology metadata** (anchor-fluvial-mineralogy)
  - Logic: `_petrology_metadata` computes feldspar/quartz/clay and cement signatures from mask means.

**Key params**: overlay strengths (`cross_bed_strength`, `ripple_strength`, `ripple_wavelength_px`, `fining_strength`, `mudstone_strength`, `lateral_accretion_strength`).


In [ ]:
inspect.getsource(gg.apply_sedimentary_overlays)


## Metrics — variogram/β/H/fractal and QA

- **Variogram & β/D/H** (anchor-fluvial-variogram)
  - Geologic intent: Texture roughness and self-similarity. β slope and derived H/D track heterogeneity.
  - Logic: `stats.compute_variogram` builds directional/isotropic semi-variograms; `fit_power_law` fits β; `fractal_dimension` maps β → D.
- **PSD anisotropy** (anchor-fluvial-psd-topology)
  - Logic: `psd_anisotropy` computes spectral ellipse (aspect, theta) to gauge directional fabrics.
- **Topology QA**
  - Logic: `topology_metrics` counts components/areas to catch mask pathologies.
- **Combined metrics payload**
  - Logic: `compute_metrics` aggregates β_dir/iso, two-segment fits (β1/β2, H0), entropy, PSD, topology, QA flags; `preview_metrics` is a lightweight subset for interactive previews.

Improvement ideas: add empirical thresholds for QA flags; compare β/H against lab/field ranges per environment.


In [ ]:
inspect.getsource(stats.compute_metrics)


## Not implemented yet — Aeolian / Estuarine

- Generators are placeholders (`generate_aeolian`, `generate_estuarine`). When implemented, mirror this rule↔code pattern: crest generation, slipfaces, interdune corridors (aeolian); ebb/flood channels, tidal bars, wave influence (estuarine); then add overlays/metrics analogously.
